# Lesson 08: Data Is King — Managing AI's Raw Ingredients

## Learning Objectives
- Understand the central role of data in AI Engineering
- Use code to identify and clean "dirty data"
- Experience using AI to synthesize training data
- Understand the basics of data augmentation and quality assessment

> Garbage In, Garbage Out. No matter how strong the model, data quality determines everything.

## Environment Setup

> Please run `00_Environment_Setup.ipynb` first to set up dependencies and API keys,
> then return to this notebook.

Once done, run the cell below to load environment variables:

In [ ]:
# Load API key from .env file (no need to enter it every time)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== Pick a provider: change this one line, nothing else =====
#   'openai'     cloud  needs OPENAI_API_KEY      strongest, has embeddings
#   'deepseek'   cloud  needs DEEPSEEK_API_KEY    cheapest cloud, no embeddings
#   'openrouter' cloud  needs OPENROUTER_API_KEY  many vendors, no embeddings
#   'ollama'     local  no key, free and offline  run `ollama serve` and pull the model first
PROVIDER = 'openai'

# All four speak the OpenAI API format. They differ only in URL, key, model names.
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = OpenAI's default endpoint
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # small model: cheap and fast
        'model_big': 'gpt-5.6-terra',                # big model: pricier and stronger
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # fast and cheap
        'model_big': 'deepseek-v4-pro',              # stronger and slower; both V4 models think first
        'embedding_model': None,                     # DeepSeek has no embeddings endpoint
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter does not proxy embeddings
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # local models ignore the key
        'model': 'gemma4:e2b-mlx',                   # run: ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # run: ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# Check the key first: with no key, the OpenAI client raises a long traceback.
if not cfg['api_key']:
    raise SystemExit(
        f"No API key for '{PROVIDER}'. Either add {PROVIDER.upper()}_API_KEY to your .env file,\n"
        f"or set PROVIDER = 'ollama' above to run locally with no key at all."
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# Every lesson below uses only these three names, so switching provider needs no
# other code change.
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'Connected! provider = {PROVIDER}, default model = {MODEL}')


---

## Activity 1: Identify and Clean "Dirty Data"

### Activity Goal
When you receive a dataset, the first step is not to use it directly — it's to check its quality.
This activity lets you identify common data issues firsthand: duplicates, contradictions, inconsistent formats, and missing values.

In [ ]:
# Activity 1: Identify dirty data

# Simulate a "Customer Feedback Dataset"
raw_data = [
    {'id': 1, 'name': 'Alice', 'rating': 5, 'comment': 'Very satisfied! Great product.', 'date': '2025-01-15'},
    {'id': 2, 'name': 'Bob', 'rating': 4, 'comment': 'Pretty good, small issue though', 'date': '2025-01-15'},
    {'id': 3, 'name': 'Alice', 'rating': 5, 'comment': 'Very satisfied! Great product.', 'date': '2025-01-15'},  # Duplicate!
    {'id': 4, 'name': 'Charlie', 'rating': 1, 'comment': 'Amazing, highly recommend!', 'date': '2025-01-16'},  # Contradiction! Low rating + positive comment
    {'id': 5, 'name': 'Diana', 'rating': 3, 'comment': '', 'date': '2025/01/16'},  # Empty comment + inconsistent date format
    {'id': 6, 'name': '', 'rating': 4, 'comment': 'Good value for money', 'date': '2025-01-17'},  # Missing name
    {'id': 7, 'name': 'Eve', 'rating': 99, 'comment': 'It is okay', 'date': '2025-01-17'},  # Rating out of range!
    {'id': 8, 'name': 'Frank', 'rating': 4, 'comment': 'great product love it', 'date': '2025-01-18'},  # Mixed content / inconsistent style
]

print('Raw dataset: total', len(raw_data), 'records\n')

# Check for issues
issues = []

# 1. Check for duplicates
seen = set()
for item in raw_data:
    key = (item['name'], item['comment'])
    if key in seen:
        issues.append(f'ID {item["id"]}: Duplicate record (name + comment matches previous)')
    seen.add(key)

# 2. Check for rating anomalies
for item in raw_data:
    if item['rating'] < 1 or item['rating'] > 5:
        issues.append(f'ID {item["id"]}: Rating {item["rating"]} out of 1-5 range')
    if item['rating'] <= 2 and any(w in item['comment'].lower() for w in ['amazing', 'great', 'recommend', 'love']):
        issues.append(f'ID {item["id"]}: Rating contradicts comment (low rating + positive comment)')

# 3. Check for missing values
for item in raw_data:
    if not item['name']:
        issues.append(f'ID {item["id"]}: Name is empty')
    if not item['comment']:
        issues.append(f'ID {item["id"]}: Comment is empty')

# 4. Check format consistency
for item in raw_data:
    if '/' in item['date']:
        issues.append(f'ID {item["id"]}: Date format inconsistent ({item["date"]}, others use 2025-01-15)')
    if item['comment'] and item['comment'].islower() and not item['comment'].endswith(('.', '!', '?')):
        issues.append(f'ID {item["id"]}: Comment style inconsistent (no capitalisation/punctuation: "{item["comment"]}")')

print('Issues found:')
for issue in issues:
    print(f'  - {issue}')

print(f'\nTotal of {len(issues)} issues found!')
print('If this data were used to train an AI, these issues would cause:')
print('  Duplicates → model overfits to certain samples')
print('  Contradictions → model learns confusing signals')
print('  Missing values → incomplete training data')
print('  Format inconsistencies → data processing errors')

### Discussion
- Does the data you use in your daily work have similar issues?
- If you have a large dataset (tens of thousands of records), how would you find these problems?
- "Data cleaning" vs "coding and tuning models" — which takes more time in practice?

---

## Activity 2: Use AI to Synthesize Training Data

### Activity Goal
AI can help you quickly generate large amounts of training data. But AI-generated data still requires human review — quality control is your responsibility.

In [ ]:
# Activity 2: AI-synthesized training data

# Scenario: Generate training data for a "Restaurant Customer Complaint AI"

data_gen_prompt = '''You are a data annotation expert. Please generate 10 simulated restaurant customer complaints.

Requirements:
1. Cover different complaint types: food quality, service attitude, wait time, cleanliness, pricing issues
2. Vary the emotional tone: from mild dissatisfaction to strong anger
3. Each complaint should include:
   - Complaint content (customer's original wording)
   - Emotion category (Mild / Moderate / Angry)
   - Expected resolution

Format:
Complaint: ...
Emotion: ...
Expected: ...'''

r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':data_gen_prompt}],
    temperature=0.8)  # Higher temperature for more diversity

print('AI-Generated Data:\n')
print(r.choices[0].message.content)

print('\n' + '='*50)
print('Data Quality Checklist:')
print('1. Are all required complaint types covered?')
print('2. Are there any unreasonable complaints? (too fake, exaggerated, unrealistic)')
print('3. Is there any bias? (stereotypes about certain foods or groups)')
print('4. Does the emotion match the complaint content?')
print('5. Are the expected resolutions reasonable?')

### Discussion
- How is the quality of the AI-generated data? Would you need to revise any of it?
- "AI-generated data + human review" vs "fully manual annotation" — what are the trade-offs?
- If the AI-generated data contains bias, what cascading effects could follow?

---

## Activity 3: Data Augmentation — Making More Out of Less

### Activity Goal
Data Augmentation creates more variants from existing data.
For text data, common augmentation techniques include: synonym replacement, paraphrasing, and back-translation.

In [ ]:
# Activity 3: Text data augmentation

original = 'The quality of this product is excellent, I am extremely satisfied, and I highly recommend it to everyone!'

# Augmentation 1: Synonym replacement
print('=== Augmentation 1: Synonym Replacement ===')
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'Replace adjectives with synonyms while preserving the meaning:\n{original}'}],
    temperature=0.5)
print(f'Original: {original}')
print(f'Variant:  {r1.choices[0].message.content}')

# Augmentation 2: Paraphrasing
print('\n=== Augmentation 2: Paraphrasing ===')
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'Express the same sentiment using a different sentence structure:\n{original}'}],
    temperature=0.5)
print(f'Original: {original}')
print(f'Variant:  {r2.choices[0].message.content}')

# Augmentation 3: Back-translation
print('\n=== Augmentation 3: Back-Translation ===')
# Translate to French first
r3_fr = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'Translate to French: {original}'}],
    temperature=0.3)
french = r3_fr.choices[0].message.content
# Translate back to English
r3_en = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'Translate to English: {french}'}],
    temperature=0.3)
print(f'Original:      {original}')
print(f'French:        {french}')
print(f'Back-translated: {r3_en.choices[0].message.content}')

print('\nOne piece of data became three — that is the power of data augmentation!')

### Discussion
- What are the pros and cons of each augmentation technique?
- Could data augmentation accidentally introduce errors?
- When should you use data augmentation? When should you avoid it?

---

## Lesson Review

| Skill | Description |
|-------|-------------|
| Dirty data identification | Spot duplicates, contradictions, missing values, format issues |
| AI data synthesis | Use AI to quickly generate training data and check its quality |
| Data augmentation | Expand datasets via synonym replacement, paraphrasing, back-translation |

### Homework
1. Find a spreadsheet you use daily and check if it has dirty data issues
2. Visit huggingface.co/datasets and browse 3 datasets that interest you
3. Use AI to generate 10 training data samples for your work/study scenario, then manually review quality